In [0]:
import os, sys
import pandas as pd
from datetime import timedelta

repo_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
sys.path.insert(0, repo_root)

from src import config_loader as cl
from src import fhir_client as fc
from src import meta

In [0]:
dbutils.widgets.removeAll()   # clears the old single-date widget
dbutils.widgets.text("resource", "ALL", "Resource (ALL or e.g. Patient)")
dbutils.widgets.text("load_dates", "", "Load dates, comma separated (blank = yesterday UTC)")
dbutils.widgets.text("max_pages", "", "Max pages override (blank = config)")
dbutils.widgets.dropdown("format", "json", ["json", "xml"], "Format")
dbutils.widgets.text("catalog", "", "Catalog override (optional)")
dbutils.widgets.text("run_id", "", "Run id (blank = generate)")
dbutils.widgets.text("pipeline_name", "manual", "Pipeline name")

resource_param = dbutils.widgets.get("resource").strip()
load_dates = [d.strip() for d in dbutils.widgets.get("load_dates").split(",") if d.strip()] \
    or [(cl.utc_now() - timedelta(days=1)).strftime("%Y-%m-%d")]
max_pages = int(dbutils.widgets.get("max_pages")) if dbutils.widgets.get("max_pages").strip() else None
fmt = dbutils.widgets.get("format")
catalog_override = dbutils.widgets.get("catalog").strip()
run_id = dbutils.widgets.get("run_id").strip() or cl.new_batch_id()
pipeline_name = dbutils.widgets.get("pipeline_name").strip()

overrides = {"project": {"catalog": catalog_override}} if catalog_override else None
cfg = cl.load_config(os.path.join(repo_root, "config", "config.yaml"), overrides)

only = None if resource_param.upper() == "ALL" else [resource_param]
resources = cl.get_resources(cfg, only)
print(f"run_id={run_id} | pipeline={pipeline_name} | format={fmt}")
print(f"dates={load_dates} | max_pages={max_pages or cfg['api']['max_pages_per_run']}")
print("Resources, in order:", [r["name"] for r in resources])

In [0]:
meta.create_meta_tables(spark, cfg)

summaries = []
for r in resources:                    # Patient -> Encounter -> Observation -> Condition
    for d in load_dates:
        print(f"Fetching {r['name']} for {d} ...")
        s = fc.ingest_resource_to_raw(cfg, r["name"], d, max_pages=max_pages, fmt=fmt)
        meta.log_raw_summary(spark, cfg, s, run_id, pipeline_name)   # logged even if it failed
        summaries.append(s)
        flag = "  <-- TRUNCATED by page cap" if s["truncated"] else ""
        print(f"  {s['status']}: {s['pages_fetched']} pages, {s['records_fetched']} records{flag}")

display(pd.DataFrame(summaries)[["resource", "load_date", "status", "pages_fetched",
                                 "records_fetched", "truncated", "error"]])

failed = [s for s in summaries if s["status"] != "SUCCESS"]
if failed:
    raise Exception(f"Ingestion failed for: {[(s['resource'], s['load_date'], s['error']) for s in failed]}")

In [0]:
s = summaries[0]
files = sorted(os.listdir(s["raw_dir"]))
print(s["raw_dir"]); print(files[:5], "...")

manifest = pd.read_json(os.path.join(s["raw_dir"], "_manifest.jsonl"), lines=True)
display(manifest[["page_number", "entry_count", "bytes", "extracted_at", "api_url"]])

# peek at the first raw page (it should start with {"resourceType":"Bundle" ...)
with open(os.path.join(s["raw_dir"], "page_0001.json"), "r") as f:
    print(f.read(600))